# Step 1. Upload csv files and transfer them to DuckDB file

Load all csv files into DuctDB tables

In [ ]:
import pandas as pd
import duckdb
import os

# Initialize DuckDB in-memory database
con = duckdb.connect(database=':memory:', read_only=False)

# Define the directory where CSV files are located
csv_dir = '/content/'

# Get a list of all files in the directory
all_files = os.listdir(csv_dir)

# Filter to include only CSV files
csv_files = [f for f in all_files if f.endswith('.csv')]

print(f"Found {len(csv_files)} CSV files in '{csv_dir}'.")

# Process each CSV file
for file_name in csv_files:
    full_path = os.path.join(csv_dir, file_name)
    try:
        # Read the CSV file into a pandas DataFrame, interpreting 'None' as nulls
        df = pd.read_csv(full_path, na_values=['None'])

        # Clean up file name for DuckDB table name (remove extension and replace invalid characters)
        table_name = file_name.rsplit('.', 1)[0].replace(' ', '_').replace('-', '_')

        # Register DataFrame as a DuckDB table
        con.execute(f"CREATE OR REPLACE TABLE {table_name} AS SELECT * FROM df")
        print(f"'{file_name}' loaded into DuckDB table: '{table_name}' with 'None' as nulls.")
    except Exception as e:
        print(f"Error processing '{file_name}': {e}")

print("\nAll available CSVs are now loaded as tables in DuckDB (if successful).")

# Verify the data loading
print("\n--- Verification ---")
print("Tables in DuckDB:")
all_tables = con.execute("SHOW TABLES").fetchall()
for table_tuple in all_tables:
    table_name = table_tuple[0]
    print(f"\nTable: {table_name}")
    print("Schema:")
    print(con.execute(f"DESCRIBE {table_name}").fetchdf())
    print("First 5 rows (including potential nulls):")
    print(con.execute(f"SELECT * FROM {table_name} LIMIT 5").fetchdf())
    print("\n")

Based on results, common columns are `customer_id`, `order_id`, `review_id`, `product_id`, `seller_i`, and `order_item_id`.

Create a SQL view named `main_orders_view` by performing LEFT JOINs on

`olist_orders_dataset`

`olist_customers_dataset`

`olist_order_items_dataset`

`olist_order_payments_dataset`

using `order_id` and `customer_id` as common columns, then verify the view creation and its schema.


In [ ]:
sql_query = """CREATE OR REPLACE VIEW main_orders_view AS
SELECT
    o.*,
    c.customer_unique_id,
    c.customer_zip_code_prefix,
    c.customer_city,
    c.customer_state,
    oi.order_item_id,
    oi.product_id,
    oi.seller_id,
    oi.shipping_limit_date,
    oi.price,
    oi.freight_value,
    op.payment_sequential,
    op.payment_type,
    op.payment_installments,
    op.payment_value
FROM olist_orders_dataset AS o
LEFT JOIN olist_customers_dataset AS c
    ON o.customer_id = c.customer_id
LEFT JOIN olist_order_items_dataset AS oi
    ON o.order_id = oi.order_id
LEFT JOIN olist_order_payments_dataset AS op
    ON o.order_id = op.order_id;
"""

con.execute(sql_query)
print("View 'main_orders_view' created successfully in DuckDB.")

# Verify the view by listing tables/views and describing it
print("\nTables and Views in DuckDB after creation:")
print(con.execute("SHOW TABLES").fetchall())
print("\nSchema of 'main_orders_view':")
print(con.execute("DESCRIBE main_orders_view").fetchall())

**Summary**

The process of creating the `main_orders_view` involved constructing a SQL query that performed `LEFT JOIN` operations on four datasets: `olist_orders_dataset`

`olist_customers_dataset`

`olist_order_items_dataset`

`olist_order_payments_dataset`.


The joins were made using `order_id` and `customer_id` as common columns.

The view's readiness for further analysis was confirmed by successfully executing the creation query, verifying its presence among database objects, and describing its schema, which accurately reflected the joined columns.



**Data Analysis Key Findings**
*   A SQL view named `main_orders_view` was successfully created in the DuckDB database using a `CREATE OR REPLACE VIEW` statement.
*   The `main_orders_view` was confirmed to be present in the database through the `SHOW TABLES` command.
*   The schema of `main_orders_view` was verified using `DESCRIBE main_orders_view`, confirming that all selected columns from the joined tables (`olist_orders_dataset`, `olist_customers_dataset`, `olist_order_items_dataset`, and `olist_order_payments_dataset`) were correctly included, indicating successful `LEFT JOIN` operations.

**Insights or Next Steps**
*   The `main_orders_view` is now available and properly structured, providing a unified dataset ready for comprehensive order-related analysis.



In [ ]:
# Calculate basic statistics from the main_orders_view
stats_query = """
SELECT
    COUNT(*) AS total_records,
    COUNT(DISTINCT order_id) AS unique_orders,
    COUNT(DISTINCT customer_unique_id) AS unique_customers
FROM main_orders_view;
"""

# Fetch the results into a pandas DataFrame
df_stats = con.execute(stats_query).fetchdf()

# Print the results
print("--- Main Orders View Statistics ---")
print(df_stats.to_string(index=False))

# Calculate and print order-to-customer ratio
ratio = df_stats['unique_orders'][0] / df_stats['unique_customers'][0]
print(f"\nAverage orders per unique customer: {ratio:.2f}")

In [ ]:
# check total records in each table
for table_tuple in all_tables:
    # Extract the name from the tuple (usually the first element)
    table_name = table_tuple[0]

    print(f"\n{'='*30}")
    print(f"Table: {table_name}")
    print(f"{'='*30}")

    print("\nSchema:")
    print(con.execute(f"DESCRIBE {table_name}").fetchdf())

    print("\nFirst 5 rows (including potential nulls):")
    # Added the closing parenthesis and .fetchdf() to complete the logic
    print(con.execute(f"SELECT * FROM {table_name} LIMIT 5").fetchdf())

# check total record of each table in all_tables
print("\n--- Total Records in Each Table ---")
# Calculate the max length for pretty alignment (optional but nice)
max_len = max(len(t[0]) for t in all_tables)

for table_tuple in all_tables:
    table_name = table_tuple[0]
    # Fetch the count
    total_records = con.execute(f"SELECT COUNT(*) FROM {table_name}").fetchone()[0]

    # Use f-string padding to align the counts
    print(f"{table_name:<{max_len + 2}} : {total_records:,}")

In [ ]:
# download this DuctDB file to local folder


# Task
Export the current in-memory DuckDB database to a persistent file named "olist_ecommerce.db" and download it to your local machine using `google.colab.files.download`.

## Export and Download DuckDB File

### Subtask:
Export the in-memory DuckDB database to a persistent file and download it to the local machine.


**Reasoning**:
I will create a persistent DuckDB database file by attaching it to the current connection and copying all tables and views into it, then initiate the download using google.colab.files.



In [ ]:
import os
from google.colab import files

# 1. Configuration
db_file = 'olist_ecommerce.db'
alias = 'persistent_db'

# 2. Cleanup: Handle existing attachments or old files
try:
    existing_dbs = con.execute("SELECT database_name FROM duckdb_databases()").fetchall()
    if any(db[0] == alias for db in existing_dbs):
        print(f"Detaching existing {alias}...")
        con.execute(f"DETACH {alias}")
except Exception:
    pass # If we can't even check databases, we'll just try to move on

if os.path.exists(db_file):
    os.remove(db_file)
    print(f"Removed old {db_file} to start fresh.")

# 3. Attach the new database file
con.execute(f"ATTACH '{db_file}' AS {alias}")

# 4. Identify objects to migrate (Legacy-Compatible Version)
# Get Tables
tables = con.execute("SELECT table_name FROM duckdb_tables() WHERE schema_name = 'main'").fetchall()
# Get Views
views = con.execute("SELECT view_name FROM duckdb_views() WHERE schema_name = 'main'").fetchall()

print(f"Found {len(tables)} tables and {len(views)} views to export.")

# 5. Migration Loop - Tables
for (table_name,) in tables:
    try:
        con.execute(f"CREATE TABLE {alias}.{table_name} AS SELECT * FROM main.{table_name}")
        print(f"  [Table] {table_name} copied.")
    except Exception as e:
        print(f"  [Error] Could not copy table {table_name}: {e}")

# 6. Migration Loop - Views
for (view_name,) in views:
    try:
        # Get the SQL definition
        view_sql = con.execute(f"SELECT sql FROM duckdb_views() WHERE view_name = '{view_name}'").fetchone()[0]
        # Recreate view (splitting at 'AS' to get the query part)
        query_part = view_sql.split('AS', 1)[1]
        con.execute(f"CREATE VIEW {alias}.{view_name} AS {query_part}")
        print(f"  [View]  {view_name} recreated.")
    except Exception as e:
        print(f"  [Error] Could not recreate view {view_name}: {e}")

# 7. Finalize and Download
con.execute(f"DETACH {alias}")
print(f"\n✅ Database successfully exported to {db_file}")

files.download(db_file)